# v2: 原始benchmark训练方式 + L2 Tick因子

使用原始benchmark的训练方式，添加L2 tick因子。

In [ ]:
import sys, json, os, gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('请从 Quant 项目目录或其子目录启动 Jupyter')
sys.path.insert(0, str(PROJECT_ROOT))
from strategies.futures.alstm.data import load_minute_bars
os.chdir(PROJECT_ROOT)

import qlib
from qlib.config import C
from qlib.data import D
from qlib.data.dataset import DatasetH, TSDatasetH
from qlib.data.dataset.handler import DataHandlerLP

from engine.backtest.futures_backtest import StrategyRuleConfig, run_backtest, shift_signal_for_execution
from engine.models.alstm import train_predict_return_and_state_alstm

C.joblib_n_jobs = 1
print("完成")

In [ ]:
# 参数（与benchmark一致）
target_instrument = "IF"
freq = "1min"
rolling_test_start = "2025-01-01"
rolling_test_end = "2025-12-31"
train_months = 12
valid_months = 6
test_months = 3
step_len = 20
account = 1_500_000
contract_multiplier = 300

# 使用原始benchmark的特征（不含L2 tick）
base_features = [
    'returns_1min', 'returns_5min', 'returns_15min', 'returns_30min',
    'momentum_5', 'momentum_10', 'momentum_20', 'momentum_30',
    'intra_mom_5', 'intra_mom_10', 'intra_mom_20', 'intra_mom_30',
    'volatility_5', 'volatility_10', 'volatility_15', 'volatility_20', 'volatility_30',
    'atr_5', 'atr_10', 'atr_20', 'atr_30',
    'sma_5', 'sma_10', 'sma_15', 'sma_20', 'sma_30',
    'sma_cross_5_10', 'sma_cross_10_20',
    'price_position_sma5', 'price_position_sma10', 'price_position_sma15', 'price_position_sma20', 'price_position_sma30',
    'volume_ratio_5', 'volume_ratio_10', 'volume_ratio_20', 'volume_ratio_30',
    'volume_persistence_5', 'volume_persistence_10', 'volume_persistence_20',
    'up_down_volatility_ratio_20',
    'bb_position_20', 'bb_width_20', 'williams_r_5', 'williams_r_20', 'rsi_20', 'cci_20', 'cmo_20', 'mfi_20',
    'vpt_sma_5', 'vpt_sma_10', 'vpt_sma_20',
    'k_percent', 'd_percent', 'vwap_ratio', 'close_open_ratio', 'high_low_ratio', 'high_close_ratio', 'low_close_ratio',
    'price_acceleration', 'short_term_momentum_volatility', 'short_term_trend_strength',
    'tick_open_interest_chg_1min', 'mid_price_5_20', 'volatility_regime_10',
]

# 添加L2 tick因子
l2_tick_features = [
    'tick_l2_spread', 'tick_l2_imbalance', 'tick_l2_mid_price',
    'tick_l2_volume_change', 'tick_l2_oi_change', 'tick_l2_price_momentum',
]

all_features = base_features + l2_tick_features

alstm_params = {
    'hidden_size': 32, 'num_layers': 2, 'dropout': 0.3,
    'n_epochs': 30, 'lr': 1e-4, 'batch_size': 128,
    'early_stop': 8, 'metric': 'loss', 'loss': 'mse',
    'optimizer': 'adam', 'GPU': 0, 'seed': 2026,
}
state_params = {
    'state_hidden_size': 32,
    'state_num_classes': 3,
    'state_class_names': {0: 'down', 1: 'flat', 2: 'up'},
    'state_lr': 5e-4,
    'state_class_weight_mode': 'sqrt_balanced',
    'state_class_weight_clip': (0.5, 3.0),
    'state_loss': 'focal',
    'state_focal_gamma': 3.0,
    'state_selection_metric': 'valid_loss',
}
state_return_threshold = 0.0005
label_horizon = 5
label_base_lag = 1

output_root = PROJECT_ROOT / 'artifacts' / 'futures_alstm' / 'alstm_v2_qlib_l2tick'
output_root.mkdir(parents=True, exist_ok=True)
print(f"特征: {len(all_features)} (基础{len(base_features)} + L2 {len(l2_tick_features)})")

In [ ]:
# 使用包含L2 tick因子的provider
provider_dir = str(PROJECT_ROOT / "data_lake/future/market_data/IF_features_with_l2tick")
print(f"Provider: {provider_dir}")

# 获取特征表达式
from factor_library.futures.minute_factors import MinuteFactorLibrary
mf = MinuteFactorLibrary.FACTORS

feature_expressions = []
for name in base_features:
    if name in mf:
        feature_expressions.append(mf[name])
for name in l2_tick_features:
    feature_expressions.append(f'${name}')

print(f"特征表达式: {len(feature_expressions)} 个")

# 滚动窗口
windows = []
current = pd.Timestamp(rolling_test_start)
while current < pd.Timestamp(rolling_test_end):
    end = current + pd.DateOffset(months=test_months)
    if end > pd.Timestamp(rolling_test_end): end = pd.Timestamp(rolling_test_end)
    valid_end = current
    valid_start = valid_end - pd.DateOffset(months=valid_months)
    train_end = valid_start
    train_start = train_end - pd.DateOffset(months=train_months)
    windows.append({'window_id': current.strftime('%Y-%m'), 'train_start': train_start.strftime('%Y-%m-%d'), 'train_end': train_end.strftime('%Y-%m-%d'), 'valid_start': valid_start.strftime('%Y-%m-%d'), 'valid_end': valid_end.strftime('%Y-%m-%d'), 'test_start': current.strftime('%Y-%m-%d'), 'test_end': end.strftime('%Y-%m-%d')})
    current = end

print(f"滚动窗口: {len(windows)} 个")

In [ ]:
# 训练（使用原始benchmark的方式）
from infra.qlib_processors import make_label_expr

raw_label_expr = f'Ref($vwap, -{label_horizon+label_base_lag})/Ref($vwap, -{label_base_lag})-1'
label_expr = make_label_expr(mode='state', horizon=label_horizon, base_lag=label_base_lag, vol_window=20, sqrt_horizon=True, up_threshold=state_return_threshold, down_threshold=-state_return_threshold, eps=1e-12)

def build_dataset_for_window(window, feature_list, label_expr):
    infer_processors = [
        {'class': 'RobustZScoreNorm', 'kwargs': {'fit_start_time': window['train_start'], 'fit_end_time': window['train_end'], 'fields_group': 'feature', 'clip_outlier': True}},
        {'class': 'Fillna', 'kwargs': {'fields_group': 'feature', 'fill_value': 0}},
    ]
    learn_processors = [{'class': 'Fillna', 'kwargs': {'fields_group': 'label', 'fill_value': 0}}]
    handler_config = {
        'class': 'DataHandlerLP', 'module_path': 'qlib.data.dataset.handler',
        'kwargs': {
            'instruments': [target_instrument], 'start_time': window['train_start'], 'end_time': window['test_end'],
            'infer_processors': infer_processors, 'learn_processors': learn_processors,
            'data_loader': {'class': 'QlibDataLoader', 'module_path': 'qlib.data.dataset.loader',
                'kwargs': {'config': {'feature': feature_list, 'label': [label_expr]}, 'freq': freq}},
        },
    }
    return DatasetH(handler=handler_config, segments={'train': (window['train_start'], window['train_end']), 'valid': (window['valid_start'], window['valid_end']), 'test': (window['test_start'], window['test_end'])})

qlib.init(provider_uri={freq: provider_dir}, region='cn')

all_results = {}
for window in windows:
    wid = window['window_id']
    print(f"\n{'='*60}")
    print(f"窗口: {wid}")
    print(f"{'='*60}")
    try:
        base_dataset = build_dataset_for_window(window, feature_expressions, label_expr)
        ts_dataset = TSDatasetH(handler=base_dataset.handler, segments=base_dataset.segments, step_len=step_len, freq=freq)
        result = train_predict_return_and_state_alstm(
            ts_dataset=ts_dataset, base_dataset=base_dataset,
            dataset_builder=lambda **kw: build_dataset_for_window(window, feature_expressions, label_expr),
            dataset_kwargs={'start_time': window['train_start'], 'end_time': window['test_end'], 'train_end': window['train_end'], 'valid_start': window['valid_start'], 'valid_end': window['valid_end'], 'test_start': window['test_start']},
            raw_label_expr=raw_label_expr, d_feat=len(feature_expressions), step_len=step_len,
            model_task='state', label_is_standardized=False,
            model_params={**alstm_params, **state_params},
            state_return_threshold=state_return_threshold,
            model_save_path=str(output_root / f'{wid}_model.pt'),
            state_model_save_path=str(output_root / f'{wid}_state.pt'),
            artifact_dir=str(output_root / wid), artifact_prefix=wid, show_progress=True,
        )
        all_results[wid] = result
        print(f"  训练完成")
    except Exception as e:
        print(f"  训练失败: {e}")
        import traceback; traceback.print_exc()

print(f"\n训练完成，共 {len(all_results)} 个窗口")